# H8 재작성 판단 신호의 분포 (정답 라벨 미사용)

EXP-011 기준 설정(D-29)의 1차 검색 결과로 신뢰도 신호를 계산하고, 임계값 후보마다 재작성 대상이 되는 질의 비율을 본다. 정답 문서는 읽지 않는다. 임계값을 정하는 근거라서 결과를 보고 판정이 기울지 않도록 라벨을 쓰지 않는다.

- A′: dense 상위 5개 문서와 BM25 상위 5개 문서의 교집합 크기(0\~5)
- B′: dense 문서 점수(청크 최댓값)의 5위−6위 차, 1위−10위 차

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from rag.index import load_cfg
from rag.search import query_embeddings, query_files, query_list, read_texts
from rag.retrieve import Retriever

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
CANDIDATES_A = [1, 2, 3]
QUANTILES_B = [0.10, 0.25, 0.50]

## 신호 계산

In [4]:
def signals(cfg):
    ret = Retriever(cfg)
    files = query_files(cfg)
    qids, text_path = query_list(cfg)
    q_emb = query_embeddings(cfg, qids, text_path, files['emb'])
    tokens = ret.query_tokens(read_texts(text_path, qids))
    rows = []
    for start in range(0, len(qids), 256):
        scores = q_emb[start:start + 256] @ ret.chunk_emb.T
        for j, row in enumerate(scores, start):
            named, _, _ = ret.lists(row, tokens[j], {})  # 신호는 dense·BM25 목록만 본다
            doc_scores = np.full(len(ret.doc_names), -np.inf, dtype=row.dtype)
            np.maximum.at(doc_scores, ret.chunk_doc, row)
            top = np.sort(doc_scores)[::-1][:10]
            rows.append({'qid': qids[j],
                         'A': len(set(named['dense'][:5]) & set(named['bm25'][:5])),
                         'B56': float(top[4] - top[5]), 'B110': float(top[0] - top[9])})
    return pd.DataFrame(rows).set_index('qid')


sig = {}
for qs in ['summary_q', 'multidoc']:
    cfg = load_cfg(['+exp=exp011', 'index.scope=full', f'search.queries={qs}'])
    sig[qs] = signals(cfg)
    print(qs, sig[qs].shape)

summary_q (3016, 3)
multidoc (600, 3)


## A′ 분포와 후보별 재작성 비율

In [5]:
for qs, s in sig.items():
    dist = s['A'].value_counts().sort_index()
    print(qs, '교집합 크기 분포', dist.to_dict())
    print('  ', {f'A<={a}': f"{(s['A'] <= a).mean():.1%}" for a in CANDIDATES_A})

summary_q 교집합 크기 분포 {0: 167, 1: 989, 2: 990, 3: 617, 4: 219, 5: 34}
   {'A<=1': '38.3%', 'A<=2': '71.2%', 'A<=3': '91.6%'}
multidoc 교집합 크기 분포 {0: 188, 1: 163, 2: 121, 3: 92, 4: 35, 5: 1}
   {'A<=1': '58.5%', 'A<=2': '78.7%', 'A<=3': '94.0%'}


## B′ 분포와 분위 임계값

In [6]:
for qs, s in sig.items():
    for col in ['B56', 'B110']:
        q = s[col].quantile(QUANTILES_B)
        print(qs, col, 'median', round(s[col].median(), 4), {f'{p:.0%}': round(v, 4) for p, v in q.items()})

summary_q B56 median 0.0046 {'10%': 0.0007, '25%': 0.0019, '50%': 0.0046}
summary_q B110 median 0.1564 {'10%': 0.0588, '25%': 0.0964, '50%': 0.1564}
multidoc B56 median 0.004 {'10%': 0.0005, '25%': 0.0016, '50%': 0.004}
multidoc B110 median 0.0789 {'10%': 0.0312, '25%': 0.0473, '50%': 0.0789}


## 두 신호를 함께 쓸 때 (A′ 또는 B′ 하위 분위)

In [7]:
s = sig['summary_q']
rows = []
for a in CANDIDATES_A:
    for p in QUANTILES_B:
        th = s['B110'].quantile(p)
        rows.append({'A<=': a, 'B110 하위': f'{p:.0%}', '재작성 비율': (s['A'] <= a) | (s['B110'] <= th)})
print(pd.DataFrame([{**{k: v for k, v in r.items() if k != '재작성 비율'}, '재작성 비율': f"{r['재작성 비율'].mean():.1%}"} for r in rows]).to_string(index=False))
print('A와 B110의 상관(spearman)', round(s['A'].corr(s['B110'], method='spearman'), 3))

 A<= B110 하위 재작성 비율
   1     10%  40.8%
   1     25%  48.3%
   1     50%  65.0%
   2     10%  71.6%
   2     25%  74.4%
   2     50%  82.1%
   3     10%  91.7%
   3     25%  92.3%
   3     50%  94.3%
A와 B110의 상관(spearman) 0.283


## 합의가 전혀 없는 경우(A′ = 0)를 쓰는 조합

A′ ≤ 1도 38%가 걸려 선택성이 낮다. 정답이 하나인 질의에서 두 목록의 상위 5개는 대부분 오답 문서라 원래 잘 겹치지 않는 것으로 보인다. 그래서 "하나도 겹치지 않음"을 따로 본다.

In [8]:
for qs, s in sig.items():
    out = {'A=0': (s['A'] == 0).mean()}
    for p in QUANTILES_B[:2]:
        th = s['B110'].quantile(p)
        out[f'B110 하위 {p:.0%}'] = (s['B110'] <= th).mean()
        out[f'A=0 또는 B110 하위 {p:.0%}'] = ((s['A'] == 0) | (s['B110'] <= th)).mean()
    print(qs, {k: f'{v:.1%}' for k, v in out.items()})

summary_q {'A=0': '5.5%', 'B110 하위 10%': '10.0%', 'A=0 또는 B110 하위 10%': '12.0%', 'B110 하위 25%': '25.0%', 'A=0 또는 B110 하위 25%': '25.5%'}
multidoc {'A=0': '31.3%', 'B110 하위 10%': '10.0%', 'A=0 또는 B110 하위 10%': '35.0%', 'B110 하위 25%': '25.0%', 'A=0 또는 B110 하위 25%': '41.8%'}
